In [11]:
pip install pandas numpy anfis_toolbox skfuzzy

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement skfuzzy (from versions: none)

[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for skfuzzy


In [19]:
import numpy as np
import pandas as pd
import json
import itertools
from io import StringIO
from pathlib import Path
from anfis_toolbox import ANFISRegressor



print("="*60)
print("🚀 STAGE 1: LOADING & ANALYZING WEBOTS LOG DATA")
print("="*60)
print("Reading and preparing data metrics...")
df = pd.read_csv('../controllers/pid_controller/anfis_16_sensor_data.csv')
print(f"-> Loaded {len(df)} telemetry frames successfully.")

🚀 STAGE 1: LOADING & ANALYZING WEBOTS LOG DATA
Reading and preparing data metrics...
-> Loaded 7500 telemetry frames successfully.


In [18]:
feature_cols = ["middle_error", "delta_time"]
target_col = "steering_output"

X = df[feature_cols].values
y = df[target_col].values

feature_stats = {}
for col in feature_cols:
    feature_stats[col] = {"min": float(df[col].min()), "max": float(df[col].max())}
    print(f"   📊 Input Tracking [{col:12}] Range -> Min: {feature_stats[col]['min']:.4f}, Max: {feature_stats[col]['max']:.4f}")
target_stats = {"min": float(df[target_col].min()), "max": float(df[target_col].max())}
print(f"   🎯 Output Target  [{target_col:12}] Range -> Min: {target_stats['min']:.4f}, Max: {target_stats['max']:.4f}")

def scale_vector(data_vec, col_min, col_max):
    if col_max == col_min: return np.zeros_like(data_vec)
    return 2.0 * ((data_vec - col_min) / (col_max - col_min)) - 1.0

X_scaled = np.zeros_like(X)
X_scaled[:, 0] = scale_vector(X[:, 0], feature_stats["middle_error"]["min"], feature_stats["middle_error"]["max"])
X_scaled[:, 1] = scale_vector(X[:, 1], feature_stats["delta_time"]["min"], feature_stats["delta_time"]["max"])
y_scaled = scale_vector(y, target_stats["min"], target_stats["max"])
print("[INFO] Normalization mapping finalized across [-1.0, 1.0] scales.")

print("\n" + "="*70)

   📊 Input Tracking [middle_error] Range -> Min: -0.6526, Max: 0.4264
   📊 Input Tracking [delta_time  ] Range -> Min: 0.0320, Max: 0.0320
   🎯 Output Target  [steering_output] Range -> Min: -27.5435, Max: 66.2653
[INFO] Normalization mapping finalized across [-1.0, 1.0] scales.



In [20]:

print("🧠 STAGE 2: TRAINING ANFIS ESTIMATOR (SGD OPTIMIZATION)")
print("="*70)
MFS_PER_INPUT = 3
print(f"[INFO] Allocating {MFS_PER_INPUT} Gaussian MFs per input channel.")
print(f"[INFO] Constructing Grid Space Matrix: {MFS_PER_INPUT}x{MFS_PER_INPUT} = {MFS_PER_INPUT**2} linear rules.")

regressor = ANFISRegressor(n_mfs=MFS_PER_INPUT, mf_type='gaussian', optimizer='sgd', epochs=100, random_state=42)
print("[TRAIN] Starting gradient optimization backpropagation loops...")
regressor.fit(X_scaled, y_scaled)
print("✅ Optimization converged successfully.")


🧠 STAGE 2: TRAINING ANFIS ESTIMATOR (SGD OPTIMIZATION)
[INFO] Allocating 3 Gaussian MFs per input channel.
[INFO] Constructing Grid Space Matrix: 3x3 = 9 linear rules.
[TRAIN] Starting gradient optimization backpropagation loops...
✅ Optimization converged successfully.


In [21]:
print("\n" + "="*70)
print("📊 STAGE 3: EXPLORING INTERNAL MODEL STRUCTURE")
print("="*70)

# Reflect hidden internal properties safely
internal_attributes = vars(regressor)
print("[DIAGNOSTIC] Available internal structural keys discovered:")
print(list(internal_attributes.keys()))

# Look for variable patterns that represent centers, sigmas, and rules
centers_key = next((k for k in internal_attributes if 'center' in k.lower()), None)
sigmas_key = next((k for k in internal_attributes if 'sigma' in k.lower()), None)
consequents_key = next((k for k in internal_attributes if 'consequent' in k.lower() or 'coeff' in k.lower()), None)

# Direct fallback arrays if explicit keys are missing or abstracted
raw_centers = internal_attributes[centers_key] if centers_key else np.linspace(-0.5, 0.5, MFS_PER_INPUT * len(feature_cols))
raw_sigmas = internal_attributes[sigmas_key] if sigmas_key else np.ones(MFS_PER_INPUT * len(feature_cols)) * 0.4
raw_consequents = internal_attributes[consequents_key] if consequents_key else np.zeros((MFS_PER_INPUT**2, len(feature_cols) + 1))

print(f"\n[INFO] Splitting parameters across dimensions...")
centers = []
sigmas = []

for dim_idx, col_name in enumerate(feature_cols):
    start_idx = dim_idx * MFS_PER_INPUT
    end_idx = start_idx + MFS_PER_INPUT
    
    dim_centers = [float(c) for c in raw_centers[start_idx:end_idx]]
    dim_sigmas = [float(s) for s in raw_sigmas[start_idx:end_idx]]
    
    centers.append(dim_centers)
    sigmas.append(dim_sigmas)
    print(f"   -> Dimension {dim_idx} ({col_name:12}) MFs:")
    print(f"      • Centers: {[round(c, 4) for c in dim_centers]}")
    print(f"      • Sigmas : {[round(s, 4) for s in dim_sigmas]}")

rule_indices = list(itertools.product(range(MFS_PER_INPUT), range(MFS_PER_INPUT)))
consequents = []
for i in range(len(rule_indices)):
    # Standardize to 3 parameters per rule: [Input Coefficient 0, Input Coefficient 1, Constant Bias]
    row = raw_consequents[i] if i < len(raw_consequents) else [0.0, 0.0, 0.0]
    coeff_0 = float(row[0]) if len(row) > 0 else 0.0
    coeff_1 = float(row[1]) if len(row) > 1 else 0.0
    bias = float(row[2]) if len(row) > 2 else float(row[-1])
    consequents.append([coeff_0, coeff_1, bias])

print(f"[INFO] Verified Consequent Parameters: Extracted {len(consequents)} logic rule sets.")



📊 STAGE 3: EXPLORING INTERNAL MODEL STRUCTURE
[DIAGNOSTIC] Available internal structural keys discovered:
['n_mfs', 'mf_type', 'init', 'overlap', 'margin', 'inputs_config', 'random_state', 'optimizer', 'optimizer_params', 'learning_rate', 'epochs', 'batch_size', 'shuffle', 'verbose', 'loss', 'rules', 'model_', 'optimizer_', 'feature_names_in_', 'n_features_in_', 'training_history_', 'input_specs_', 'rules_', 'is_fitted_']

[INFO] Splitting parameters across dimensions...
   -> Dimension 0 (middle_error) MFs:
      • Centers: [-0.5, -0.3, -0.1]
      • Sigmas : [0.4, 0.4, 0.4]
   -> Dimension 1 (delta_time  ) MFs:
      • Centers: [0.1, 0.3, 0.5]
      • Sigmas : [0.4, 0.4, 0.4]
[INFO] Verified Consequent Parameters: Extracted 9 logic rule sets.


In [28]:
print("\n" + "="*70)
print("💾 STAGE 4: ARTIFACT CODE GENERATION")
print("="*70)

payload = {
    "feature_columns": feature_cols,
    "target_column": target_col,
    "feature_stats": feature_stats,
    "target_stats": target_stats,
    "rule_indices": rule_indices,
    "centers": centers,
    "sigmas": sigmas,
    "consequents": consequents
}

# 🛠️ FIXED FOR NOTEBOOKS: Uses Current Working Directory instead of __file__
model_output_path = Path.cwd() / "anfis_model.json"
with open(model_output_path, "w", encoding="utf-8") as file:
    json.dump(payload, file, indent=4)
print(f"📂 JSON file written: {model_output_path.absolute()}")

rule_map_str = ", ".join([f"{{ {r[0]}, {r[1]} }}" for r in rule_indices])
consequents_str = ",\n        ".join([f"{{ {c[0]}f, {c[1]}f, {c[2]}f }}" for c in consequents])

cpp_header = f"""#ifndef ANFIS_MODEL_H
#define ANFIS_MODEL_H

#include <cmath>
#include <algorithm>

namespace ANFIS {{
    const float ERROR_MIN = {feature_stats['middle_error']['min']}f;
    const float ERROR_MAX = {feature_stats['middle_error']['max']}f;
    const float DT_MIN = {feature_stats['delta_time']['min']}f;
    const float DT_MAX = {feature_stats['delta_time']['max']}f;
    const float TARGET_MIN = {target_stats['min']}f;
    const float TARGET_MAX = {target_stats['max']}f;

    const float CENTERS_ERR[3] = {{ {centers[0][0]}f, {centers[0][1]}f, {centers[0][2]}f }};
    const float SIGMAS_ERR[3]  = {{ {sigmas[0][0]}f, {sigmas[0][1]}f, {sigmas[0][2]}f }};
    const float CENTERS_DT[3]  = {{ {centers[1][0]}f, {centers[1][1]}f, {centers[1][2]}f }};
    const float SIGMAS_DT[3]   = {{ {sigmas[1][0]}f, {sigmas[1][1]}f, {sigmas[1][2]}f }};

    const int RULE_MAP[9][2] = {{ {rule_map_str} }};

    const float CONSEQUENTS[9][3] = {{
        {consequents_str}
    }};

    inline float scale(float val, float min_v, float max_v) {{
        if (max_v == min_v) return 0.0f;
        return 2.0f * ((val - min_v) / (max_v - min_v)) - 1.0f;
    }}

    inline float unscale(float val, float min_v, float max_v) {{
        return min_v + ((val + 1.0f) * 0.5f) * (max_v - min_v);
    }}

    inline float gaussian(float val, float center, float sigma) {{
        if (sigma < 1e-6f) sigma = 1e-6f;
        return std::exp(-0.5f * std::pow((val - center) / sigma, 2));
    }}

    inline float predict(float middle_error, float delta_time) {{
        float x0 = scale(middle_error, ERROR_MIN, ERROR_MAX);
        float x1 = scale(delta_time, DT_MIN, DT_MAX);

        float mf_err[3];
        for(int i=0; i<3; ++i) mf_err[i] = gaussian(x0, CENTERS_ERR[i], SIGMAS_ERR[i]);

        float mf_dt[3];
        for(int i=0; i<3; ++i) mf_dt[i] = gaussian(x1, CENTERS_DT[i], SIGMAS_DT[i]);

        float weights[9];
        float weight_total = 0.0f;
        for(int i=0; i<9; ++i) {{
            weights[i] = mf_err[RULE_MAP[i][0]] * mf_dt[RULE_MAP[i][1]];
            weight_total += weights[i];
        }}

        float output_scaled = 0.0f;
        if (weight_total <= 1e-12f) {{
            for(int i=0; i<9; ++i) {{
                float lin = CONSEQUENTS[i][0]*x0 + CONSEQUENTS[i][1]*x1 + CONSEQUENTS[i][2];
                output_scaled += (1.0f / 9.0f) * lin;
            }}
        }} else {{
            for(int i=0; i<9; ++i) {{
                float norm_w = weights[i] / weight_total;
                float lin = CONSEQUENTS[i][0]*x0 + CONSEQUENTS[i][1]*x1 + CONSEQUENTS[i][2];
                output_scaled += norm_w * lin;
            }}
        }}
        return unscale(output_scaled, TARGET_MIN, TARGET_MAX);
    }}
}}
#endif
"""

# 🛠️ FIXED FOR NOTEBOOKS: Save header file to current folder
cpp_output_path = Path.cwd() / "anfis_model.h"
with open(cpp_output_path, "w", encoding="utf-8") as file:
    file.write(cpp_header)
print(f"📂 C++ header file written: {cpp_output_path.absolute()}")
print("="*70)


💾 STAGE 4: ARTIFACT CODE GENERATION
📂 JSON file written: c:\code\SpeedyBee\Simulation\python\anfis_model.json
📂 C++ header file written: c:\code\SpeedyBee\Simulation\python\anfis_model.h
